# COVID-19 Clinical Trials — Exploratory Data Analysis
**Dataset:** 5,783 COVID-19 clinical trials from ClinicalTrials.gov  
**Tools:** pandas, matplotlib, seaborn  
**Goal:** Understand trial landscape — status, phases, interventions, funding, enrollment trends

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120
plt.rcParams['font.family'] = 'sans-serif'

df = pd.read_csv('covid_trials_clean.csv', low_memory=False)
print(f'Dataset shape: {df.shape}')
df.head(3)

## 1. Dataset overview & missing values

In [ ]:
print('Shape:', df.shape)
print('\nMissing values (top columns):')
print(df.isnull().sum().sort_values(ascending=False).head(10))
print('\nDate range:', pd.to_datetime(df['Start Date'], errors='coerce').min().date(),
      'to', pd.to_datetime(df['Start Date'], errors='coerce').max().date())

## 2. Trial status breakdown

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
status_counts = df['Status'].value_counts()
colors = ['#1D9E75' if s == 'Completed' else '#378ADD' if s == 'Recruiting' else '#888780'
          for s in status_counts.index]
bars = ax.barh(status_counts.index[::-1], status_counts.values[::-1], color=colors[::-1])
for bar, val in zip(bars, status_counts.values[::-1]):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height()/2,
            f'{val:,}', va='center', fontsize=10)
ax.set_xlabel('Number of trials')
ax.set_title('Trial status distribution', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/01_status_distribution.png', bbox_inches='tight')
plt.show()

## 3. Trials registered per month (2020–2021)

In [ ]:
df['start_month_dt'] = pd.to_datetime(df['start_month'], errors='coerce')
monthly = df[df['start_year'].isin([2020, 2021])].groupby('start_month').size().reset_index(name='count')
monthly = monthly.sort_values('start_month')

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(monthly)), monthly['count'], color='#378ADD', width=0.7)
ax.set_xticks(range(len(monthly)))
ax.set_xticklabels(monthly['start_month'], rotation=45, ha='right', fontsize=9)
ax.set_ylabel('Trials registered')
ax.set_title('COVID-19 trials registered per month (2020–2021)', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/02_monthly_registrations.png', bbox_inches='tight')
plt.show()

## 4. Trial phases

In [ ]:
phase_df = df[~df['Phases'].isin(['Unknown'])]['Phases'].value_counts()

fig, ax = plt.subplots(figsize=(8, 4))
bars = ax.bar(phase_df.index, phase_df.values, color='#7F77DD', width=0.6)
for bar, val in zip(bars, phase_df.values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 10,
            str(val), ha='center', fontsize=10)
ax.set_ylabel('Number of trials')
ax.set_xlabel('Phase')
ax.set_title('Trials by phase (excl. unknown)', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig('outputs/03_phase_distribution.png', bbox_inches='tight')
plt.show()

## 5. Intervention types

In [ ]:
interv = df['Intervention Type'].value_counts().head(8)

fig, ax = plt.subplots(figsize=(8, 4))
colors_i = ['#D85A30' if i == 'Drug' else '#1D9E75' if i == 'Biological' else '#888780'
            for i in interv.index]
ax.barh(interv.index[::-1], interv.values[::-1], color=colors_i[::-1])
for i, (idx, val) in enumerate(zip(interv.index[::-1], interv.values[::-1])):
    ax.text(val + 10, i, str(val), va='center', fontsize=10)
ax.set_xlabel('Number of trials')
ax.set_title('Trials by intervention type', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/04_intervention_types.png', bbox_inches='tight')
plt.show()

## 6. Enrollment distribution (cleaned)

In [ ]:
enroll = df[(df['Enrollment'] > 0) & (df['Enrollment'] < 5000)]['Enrollment']

fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(enroll, bins=50, color='#378ADD', edgecolor='white', linewidth=0.5)
ax.axvline(enroll.median(), color='#D85A30', linestyle='--', linewidth=1.5, label=f'Median: {int(enroll.median())}')
ax.set_xlabel('Enrollment count')
ax.set_ylabel('Number of trials')
ax.set_title('Enrollment distribution (0–5,000 participants)', fontsize=13, fontweight='bold', pad=12)
ax.legend()
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/05_enrollment_distribution.png', bbox_inches='tight')
plt.show()

## 7. Funding sources

In [ ]:
fund = df['Funding Simple'].value_counts()

fig, ax = plt.subplots(figsize=(6, 4))
colors_f = ['#888780','#D85A30','#378ADD','#1D9E75']
wedges, texts, autotexts = ax.pie(fund.values, labels=fund.index, autopct='%1.1f%%',
                                   colors=colors_f, startangle=90,
                                   wedgeprops={'linewidth': 0.5, 'edgecolor': 'white'})
for t in autotexts:
    t.set_fontsize(10)
ax.set_title('Funding source breakdown', fontsize=13, fontweight='bold', pad=12)
plt.tight_layout()
plt.savefig('outputs/06_funding_sources.png', bbox_inches='tight')
plt.show()

## 8. Completion rate by phase

In [ ]:
phase_completion = df[df['Phases'].isin(['Phase 1','Phase 2','Phase 3','Phase 4'])]\
    .groupby('Phases')['Is Completed'].mean().mul(100).round(1).reset_index()
phase_completion.columns = ['Phase', 'Completion Rate (%)']
print(phase_completion)

fig, ax = plt.subplots(figsize=(7, 4))
bars = ax.bar(phase_completion['Phase'], phase_completion['Completion Rate (%)'],
              color=['#CECBF6','#AFA9EC','#7F77DD','#534AB7'], width=0.5)
for bar, val in zip(bars, phase_completion['Completion Rate (%)']):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{val}%', ha='center', fontsize=11, fontweight='bold')
ax.set_ylabel('Completion rate (%)')
ax.set_ylim(0, 35)
ax.set_title('Completion rate by trial phase', fontsize=13, fontweight='bold', pad=12)
ax.spines[['top','right']].set_visible(False)
plt.tight_layout()
plt.savefig('outputs/07_completion_by_phase.png', bbox_inches='tight')
plt.show()

## 9. Key findings summary

| Finding | Value |
|---------|-------|
| Total trials | 5,783 |
| Peak registration month | April 2020 (COVID surge) |
| Most common status | Recruiting (48.5%) |
| Most common intervention | Drug trials (1,528) |
| Median enrollment | 160 participants |
| Phase 3 completion rate | ~23% (data snapshot in 2021) |
| Majority funded by | Other/non-profit (77.6%) |

### Insights for your README
1. **Speed of response:** 4,245 trials registered in 2020 alone — an unprecedented global mobilisation
2. **Drug-first approach:** Drug trials dominate (26%), followed by biological (vaccines/antibodies) at 10%
3. **Small trials:** Median enrollment of 160 suggests most are early-phase feasibility studies
4. **Completion lag:** Only ~18% of all trials show as completed — reflects the ongoing nature of the pandemic at time of data collection